# Visualizing enVision data

In [1]:
import polars as pl
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
import string
from tqdm import tqdm
import numpy as np

rows = list(string.ascii_uppercase[:16])  # A-P
cols = list(range(1, 25))                 # 1-24

wells = [f"{r}{c:02d}" for r in rows for c in cols]


dataset_path = Path(
    "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
    "Study/enVision/enVision_preprocessed/2026-04-01._."
    "PREPROCESSED.csv"
    )

raw_data_path = Path(
    "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live"
    " Study/enVision/enVision_formatted/2026-04-01._.FORMATTED.tsv"
    )

platemap = (
    pl.read_excel(
    "/home/derfelt/PartchLabFiles/Circadian"
    " Rhythms/Live Study/2026-04-01Echo_transfer.xlsx", engine="openpyxl")
    .select(["384_Well", 
             "MoleculeID", 
             "strain",
             "dex"])
    .unique(subset=["384_Well"])
    .with_columns(
        pl.col("MoleculeID").str.strip_chars()
    )
    )

well_excluded = set(wells) - set(platemap["384_Well"].to_list())

print(f"Number of wells not in platemap: {len(well_excluded)}")
# print(*well_excluded, sep="\n")

display(platemap)
# for name, df in platemap.group_by(["MoleculeID", "strain", "dex"]):
#     print(name)
#     display(df)
    # if name[0] == "LY 5uM":
    #     print(name)
    #     display(df)

Number of wells not in platemap: 146


384_Well,MoleculeID,strain,dex
str,str,str,str
"""G13""","""0.1% DMSO""","""Bmal1""","""NoSynhronisation"""
"""G12""","""MH-13 50uM""","""Bmal1""","""dex"""
"""H10""","""LY 10uM""","""Bmal1""","""dex"""
"""L15""","""MK-1804 25uM""","""Per2""","""dex"""
"""C09""","""MK-0761 25uM""","""Bmal1""","""dex"""
…,…,…,…
"""F12""","""MK-1804 100uM""","""Bmal1""","""dex"""
"""C08""","""MH-11 25uM""","""Bmal1""","""dex"""
"""G05""","""0.05% DMSO""","""Bmal1""","""NoSynhronisation"""


In [2]:
def make_title(row: dict, well_col: str, meta_cols: list[str]) -> str:
    parts = [row[well_col]]
    for col in meta_cols:
        val = row.get(col)
        if val is not None and str(val).strip() != '' and str(val).lower() != 'nan':
            parts.append(str(val))
    return ' | '.join(parts)

In [7]:
raw_timeseries = pl.read_csv(raw_data_path, separator="\t")
raw_timeseries = raw_timeseries.join(platemap, left_on="Well", right_on="384_Well", how="left")

wells = raw_timeseries["Well"].unique().sort().to_list()
out_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/enVision_visual_results/raw_timeseries.pdf")

timepoint_col = "Timepoint"
value = "CalcResultI"
meta_cols = ['MoleculeID', 'strain', 'dex']

with PdfPages(out_path) as pdf:
    g = sns.FacetGrid(
        data=raw_timeseries,
        col="Well",
        col_wrap=24,
        height=2.5,
        aspect=1.8,
        sharey=False,
        sharex=True,
    )
    g.map_dataframe(sns.lineplot, x=timepoint_col, y=value, color="red", linewidth=1.5)
    
    # add marks and set ticks after mapping
    x_min = raw_timeseries['Timepoint'].min()
    x_max = raw_timeseries['Timepoint'].max()
    tick_marks = np.arange(np.ceil(x_min / 12) * 12, x_max + 12, 12)

    for ax in g.axes.flat:
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.set_xticks(tick_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in tick_marks], fontsize=6)
        for mark in tick_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
        
        
    g.set_titles(col_template='Well {col_name}')
    g.set_axis_labels('Time (hours)', 'Detrended signal')
    g.tight_layout()
    pdf.savefig(g.figure)
    plt.close()
    
    
    for well in tqdm(wells, total=len(wells), desc="Well progress"):
        well_df = (raw_timeseries
              .filter((pl.col("Well") == well))
              .sort("Timepoint", descending=False)
              .to_pandas()
              )
        
        fig, ax = plt.subplots(figsize=(12, 4))
        
        # get first row for metadata
        row = well_df.iloc[0].to_dict()
        title = make_title(row, well_col='Well', meta_cols=meta_cols)
        
        sns.lineplot(data=well_df, x=timepoint_col, y=value, ax=ax, color="red", linewidth=2, markers=False, marker="o", markersize=2)
        
        ax.set_title(title)
        ax.set_xlabel("Timepoint")
        ax.set_ylabel("Expression")
        
        x_min = well_df[timepoint_col].min()
        x_max = well_df[timepoint_col].max()
        period_marks = np.arange(np.ceil(x_min/12) * 12, x_max + 12, 12)
        
        for mark in period_marks:
            ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
        ax.set_xticks(period_marks)
        ax.set_xticklabels([f'{int(m)}h' for m in period_marks])
        
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
        

Well progress: 100%|██████████| 384/384 [00:40<00:00,  9.39it/s]
